In [2]:
import pandas as pd
import easygui
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
from numpy.random import random
from plotly.subplots import make_subplots

In [16]:
PQ = easygui.fileopenbox(filetypes="*.txt",multiple=True)

In [17]:
PQ

['C:\\Users\\sadykpayev\\Desktop\\Джуниор\\К3\\СТР\\Моделирование СТР в случае отказа двух ЖК\\data for ML\\liquidSystem1.csv',
 'C:\\Users\\sadykpayev\\Desktop\\Джуниор\\К3\\СТР\\Моделирование СТР в случае отказа двух ЖК\\data for ML\\liquidSystem2.csv',
 'C:\\Users\\sadykpayev\\Desktop\\Джуниор\\К3\\СТР\\Моделирование СТР в случае отказа двух ЖК\\data for ML\\NorthPanelAll.csv',
 'C:\\Users\\sadykpayev\\Desktop\\Джуниор\\К3\\СТР\\Моделирование СТР в случае отказа двух ЖК\\data for ML\\payload.csv',
 'C:\\Users\\sadykpayev\\Desktop\\Джуниор\\К3\\СТР\\Моделирование СТР в случае отказа двух ЖК\\data for ML\\SouthPanelAll.csv']

In [18]:
data_load = easygui.fileopenbox(filetypes="*.txt",multiple=True)

In [23]:
data = defaultdict(list)

### before failure

In [24]:
def collecting():
    for name in data_load:
        csv = pd.DataFrame()
        csv[name.split('-')[1]] = pd.read_csv(name)['max']
        csv['payload'] = pd.read_csv(PQ[3])['load'].iloc[:126]
        csv['north'] = pd.read_csv(PQ[2])['max'].iloc[:126]
        csv['south'] = pd.read_csv(PQ[4])['max'].iloc[:126]
        csv['LiquidSystem1'] = pd.read_csv(PQ[0])['Значение'].iloc[:126]
        csv['LiquidSystem2'] = pd.read_csv(PQ[1])['Значение'].iloc[:126]
        data[name.split('-')[1]].append(csv)

In [25]:
collecting()

In [28]:
listOk = []
for name in list(data.keys()):
    if data[name][0].corr().iloc[0,1] > 0.66:
                corrPlot = px.imshow(data[name][0].corr(), title = 'Коэффициент корреляции Пирсона', text_auto = True)
                corrPlot.write_html(f'Корреляция\\payload 0.66-1 {name}.html')
                listOk.append(name)
        
    if data[name][0].corr().iloc[0,2] > 0.66:
                corrPlot = px.imshow(data[name][0].corr(), title = 'Коэффициент корреляции Пирсона', text_auto = True)
                corrPlot.write_html(f'Корреляция\\north 0.66-1 {name}.html')        
                listOk.append(name)
    if data[name][0].corr().iloc[0,3] > 0.66:
                corrPlot = px.imshow(data[name][0].corr(), title = f'Коэффициент корреляции Пирсона', text_auto = True)
                corrPlot.write_html(f'Корреляция\\south 0.66-1 {name}.html')
                listOk.append(name)
    #if 0.33 < data[name][0].corr().iloc[0,1] <= 0.66:
                #corrPlot = px.imshow(data[name][0].corr(), title = 'Коэффициент корреляции Пирсона', text_auto = True)
                #corrPlot.write_html(f'Корреляция\\payload 0.33-0.66 {name}.html')
        
    #if 0.33 < data[name][0].corr().iloc[0,2] <= 0.66:
                #corrPlot = px.imshow(data[name][0].corr(), title = 'Коэффициент корреляции Пирсона', text_auto = True)
                #corrPlot.write_html(f'Корреляция\\north 0.33-0.66 {name}.html')      

    #if 0.33 < data[name][0].corr().iloc[0,3] <= 0.66:
                #corrPlot = px.imshow(data[name][0].corr(), title = 'Коэффициент корреляции Пирсона', text_auto = True)
                #corrPlot.write_html(f'Корреляция\\south 0.33-0.66 {name}.html')


    #if 0 < data[name][0].corr().iloc[0,1] <= 0.33:
                #corrPlot = px.imshow(data[name][0].corr(), title = 'Коэффициент корреляции Пирсона', text_auto = True)
                #corrPlot.write_html(f'Корреляция\\payload 0-0.33 {name}.html')
        
    #if 0 < data[name][0].corr().iloc[0,2] <= 0.33:
                #corrPlot = px.imshow(data[name][0].corr(), title = 'Коэффициент корреляции Пирсона', text_auto = True)
                #corrPlot.write_html(f'Корреляция\\north 0-0.33 {name}.html')      

    #if 0 < data[name][0].corr().iloc[0,3] <= 0.33:
                #corrPlot = px.imshow(data[name][0].corr(), title = 'Коэффициент корреляции Пирсона', text_auto = True)
                #corrPlot.write_html(f'Корреляция\\south 0-0.33 {name}.html')

In [29]:
out = [x for x in list(data.keys()) if x not in listOk] 

In [30]:
for name in out:
    corrPlot = px.imshow(data[name][0].corr(), title = 'Коэффициент корреляции Пирсона', text_auto = True)
    corrPlot.write_html(f'Корреляция\\{name}.html')

### Machine learn before failure

In [31]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score

In [32]:
mseComparison = defaultdict(list)

In [33]:
#Linear
i_list = []
mse_list = []
r2_list = []
for i in range(len(data.keys())):
    X = data[list(data.keys())[i]][0][['payload','north','south', 'LiquidSystem1', 'LiquidSystem2']]
    y = data[list(data.keys())[i]][0][[list(data.keys())[i]]]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
    
    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
                 
    mseComparison['line'].append(mse)
                  
    if mse < 2:
        i_list.append(i)
        mse_list.append(mse)
        r2_list.append(r2)
# How much errors
len(data.keys()) - len(i_list) 

95

In [34]:
#Random Forest
i_list = []
mse_list = []
r2_list = []

i_list_out = []
mse_list_out = []
r2_list_out = []

for i in range(len(data.keys())):
    X = data[list(data.keys())[i]][0][['payload','north','south', 'LiquidSystem1', 'LiquidSystem2']]
    y = data[list(data.keys())[i]][0][[list(data.keys())[i]]]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
    
    rf_model = RandomForestRegressor(n_estimators=300, random_state=42)
    rf_model.fit(X_train, y_train.values.ravel())
    y_pred_rf = rf_model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred_rf )
    r2 = r2_score(y_test, y_pred_rf)
    mseComparison['random'].append(mse)
    if mse < 2.5:
        i_list.append(i)
        mse_list.append(mse)
        r2_list.append(r2)

    else:
        i_list_out.append(i)
        mse_list_out.append(mse)
        r2_list_out.append(r2)
# How much errors
len(data.keys()) - len(i_list) 

47

In [35]:
#Polinomial
i_list = []
mse_list = []
r2_list = []
for i in range(len(data.keys())):
    X = data[list(data.keys())[i]][0][['payload','north','south', 'LiquidSystem1', 'LiquidSystem2']]
    y = data[list(data.keys())[i]][0][[list(data.keys())[i]]]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    #полиномиальная регрессия 4-й степени
    model_poly = make_pipeline(PolynomialFeatures(degree=4), LinearRegression())
    model_poly.fit(X_train, y_train)
    y_pred_poly = model_poly.predict(X_test)
    mse = mean_squared_error(y_test, y_pred_poly)
    r2 = r2_score(y_test, y_pred_poly)
    mseComparison['poly'].append(mse)
    
    if mse < 2.5:
        i_list.append(list(data.keys())[i])
        mse_list.append(mse)
        r2_list.append(r2)
# How much errors
len(data.keys()) - len(i_list) 

88

In [36]:
#Decision Tree
i_list = []
mse_list = []
r2_list = []
for i in range(len(data.keys())):
    X = data[list(data.keys())[i]][0][['payload','north','south', 'LiquidSystem1', 'LiquidSystem2']]
    y = data[list(data.keys())[i]][0][[list(data.keys())[i]]]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    tree_reg = DecisionTreeRegressor(random_state=42)
    tree_reg.fit(X_train, y_train)
    y_pred_dt = tree_reg.predict(X_test)
    mse = mean_squared_error(y_test, y_pred_dt)
    r2 = r2_score(y_test, y_pred_dt)
    mseComparison['dtree'].append(mse)
    
    if mse < 2.5:
        i_list.append(i)
        mse_list.append(mse)
        r2_list.append(r2)
# How much errors
len(data.keys()) - len(i_list) 

81

### others method

In [248]:
# Support Vector Regression (SVR) с масштабированием
i_list = []
mse_list = []
r2_list = []

from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor

for i in range(len(data.keys())):
    X = data[list(data.keys())[i]][0][['payload','north','south', 'LiquidSystem1', 'LiquidSystem2']]
    y = data[list(data.keys())[i]][0][[list(data.keys())[i]]]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    tree_reg = DecisionTreeRegressor(random_state=42)
    tree_reg.fit(X_train, y_train)
    y_pred_dt = tree_reg.predict(X_test)
    mse = mean_squared_error(y_test, y_pred_dt)
    r2 = r2_score(y_test, y_pred_dt)
    
    if mse < 2.5:
        i_list.append(i)
        mse_list.append(mse)
        r2_list.append(r2)
# How much errors
len(data.keys()) - len(i_list) 

75

### data combining

In [37]:
line = []
poly = []
dtree = []
random = []
for x in range(len(mseComparison['line'])):
    if mseComparison['line'][x] < mseComparison['poly'][x] and mseComparison['line'][x] < mseComparison['dtree'][x] and mseComparison['line'][x] < mseComparison['random'][x]:
            line.append(x)
    if mseComparison['poly'][x] < mseComparison['line'][x] and mseComparison['poly'][x] < mseComparison['dtree'][x] and mseComparison['poly'][x] < mseComparison['random'][x]:
            poly.append(x)
    if mseComparison['dtree'][x] < mseComparison['line'][x] and mseComparison['dtree'][x] < mseComparison['poly'][x] and mseComparison['dtree'][x] < mseComparison['random'][x]:
            dtree.append(x)
    if mseComparison['random'][x] < mseComparison['line'][x] and mseComparison['random'][x] < mseComparison['dtree'][x] and mseComparison['random'][x] < mseComparison['poly'][x]:
            random.append(x)

In [38]:
mseFull = [mseComparison['line'][x] for x in line] + [mseComparison['poly'][x] for x in poly] + [mseComparison['dtree'][x] for x in dtree] + [mseComparison['random'][x] for x in random]

In [39]:
len([list(data.keys())[x] for x in line])

4

In [40]:
len([list(data.keys())[x] for x in poly])

5

In [41]:
len([list(data.keys())[x] for x in dtree])

4

In [42]:
len([list(data.keys())[x] for x in random])

94

### Saving models

In [249]:
import joblib

In [43]:
countLine = []
countPoly = []
countDtree = []
countRandom = []

In [44]:
#Linear
for i in line:
    X = data[list(data.keys())[i]][0][['payload','north','south', 'LiquidSystem1', 'LiquidSystem2']]
    y = data[list(data.keys())[i]][0][[list(data.keys())[i]]]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42) 
    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    if mse < 2.5:
        countLine.append(i)
    #joblib.dump(model, f'data for ML\\models\\Linear {list(data.keys())[i]}')
#Polynomial
for i in poly:
    X = data[list(data.keys())[i]][0][['payload','north','south', 'LiquidSystem1', 'LiquidSystem2']]
    y = data[list(data.keys())[i]][0][[list(data.keys())[i]]]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) 
    model_poly = make_pipeline(PolynomialFeatures(degree=4), LinearRegression())
    model_poly.fit(X_train, y_train)
    y_pred_poly = model_poly.predict(X_test)
    mse = mean_squared_error(y_test, y_pred_poly)
    r2 = r2_score(y_test, y_pred_poly)
    if mse < 2.5:
        countPoly.append(i)
    #joblib.dump(model_poly, f'data for ML\\models\\Polynomial {list(data.keys())[i]}')
#Decision Tree
for i in dtree:
    X = data[list(data.keys())[i]][0][['payload','north','south', 'LiquidSystem1', 'LiquidSystem2']]
    y = data[list(data.keys())[i]][0][[list(data.keys())[i]]]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) 
    tree_reg = DecisionTreeRegressor(random_state=42)
    tree_reg.fit(X_train, y_train)
    y_pred_dt = tree_reg.predict(X_test)
    mse = mean_squared_error(y_test, y_pred_dt)
    r2 = r2_score(y_test, y_pred_dt)
    if mse < 2.5:
        countDtree.append(i)
    #joblib.dump(model_poly, f'data for ML\\models\\Decision Tree {list(data.keys())[i]}')
#Random forest
for i in random:
    X = data[list(data.keys())[i]][0][['payload','north','south', 'LiquidSystem1', 'LiquidSystem2']]
    y = data[list(data.keys())[i]][0][[list(data.keys())[i]]]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42) 
    rf_model = RandomForestRegressor(n_estimators=300, random_state=42)
    rf_model.fit(X_train, y_train.values.ravel())
    y_pred_rf = rf_model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred_rf )
    r2 = r2_score(y_test, y_pred_rf)
    if mse < 2.5:
        countRandom.append(i)
    #joblib.dump(model_poly, f'data for ML\\models\\Random forest {list(data.keys())[i]}')

In [53]:
len(countRandom) + len(countPoly) + len(countLine) + len(countDtree)

62

In [46]:
len(poly) - countPoly

TypeError: unsupported operand type(s) for -: 'int' and 'list'

In [307]:
[x for x in mseFull  if x > 2.5 ]

[3.9148268983600407,
 3.0066434949776415,
 3.871209609199499,
 2.5139256074558585,
 2.5060782473653576,
 2.6795997172532307,
 11.355771047248718,
 11.357851171995687,
 7.603415578947839,
 9.0078720000003,
 3.3143717173099563,
 2.9395058771930094,
 2.9180215263157967,
 2.7740548421053055,
 3.15322088888885,
 4.370985631578955,
 5.965065608187087,
 4.732578163742849,
 3.010691309941459,
 3.01843178549703,
 3.7864286185379648]

In [47]:
#Random
[f'{list(data.keys())[x]}, {mseComparison['random'][x]}' for x in [x for x in random if x not in countRandom]]

['Т1КАНТ, 7.357224421052936',
 'Т1КИС, 2.6848988947367713',
 'Т1ПЗВ, 4.6908768683040885',
 'Т1ПОЗ, 2.850064445731138',
 'Т1ЮП, 3.2923936374269136',
 'Т2КАНТ, 6.09007831578941',
 'Т2КИС, 2.561874824561316',
 'Т2ЛК, 2.8396894444444265',
 'Т2ПЗВ, 3.2412576935672703',
 'Т2СП, 3.153833076023313',
 'Т3ЛК, 3.200476532163737',
 'Т3ЮП, 2.8646402807017246',
 'Т4ЛК, 2.9967037660819016',
 'Т4ЮП, 4.534891087719449',
 'Т5ЛК, 2.5142285614035402',
 'Т5СП, 2.6130288421050913',
 'Т5ЮП, 2.8572295438596678',
 'Т6ЛК, 4.9942687836258',
 'Т6ЮП, 6.288296766081892',
 'Т7ЛК, 6.391394643274967',
 'Т7ЮП, 3.8107160760233834',
 'Т8СП, 3.151701766081716',
 'Т9ЛК, 3.0651702865497263',
 'Т9СП, 2.631501532163725',
 'Т13СП, 2.7636146198829348',
 'Т15ЛК, 3.929664812865665',
 'Т16ЛК, 2.568393146198868',
 'Т18ЛК, 2.501473976608227',
 'Т24БМ, 2.7347676432748247',
 'Т25БМ, 3.7859133625730843',
 'Т26БМ, 7.8487591221052595',
 'Т27БМ, 7.838656002923986',
 'Т30ЛК, 2.809620432748446',
 'Т30ЮПП, 3.1741155321637744',
 'Т32ЛК, 4.109

In [48]:
#Poly
[f'{list(data.keys())[x]}, {mseComparison['poly'][x]}' for x in [x for x in poly if x not in countPoly]]

['Т1ЛК, 5.740329211784921',
 'Т2ПОЗ, 3.6346905894247534',
 'Т12ЛК, 2.763427307095832',
 'Т31ЛК, 4.363018479668328']

In [322]:
11**(0.5)

3.3166247903554

In [ ]:
#mse <2.5
[f'name: {list(data.keys())[x]}, mse: {round(mse_list[y], 2)}' for x, y in zip(i_list, range(len(mse_list)))]

In [ ]:
#mse >2.5
[f'name: {list(data.keys())[x]}, mse: {round(mse_list_out[y],2)}' for x, y in zip(i_list_out, range(len(mse_list_out)))]